In [ ]:
## Notebook 4 — Écriture Parquet (et chargement PostgreSQL bonus)

In [1]:
# Q33 — Relire le Parquet - Relire le fichier Parquet et vérifier que le nombre de lignes est identique à l'original. Afficher le schema — observer que les types sont préservés

from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("Notebook 4 - Q33 Parquet") \
    .getOrCreate()


parquet_path = "/home/jovyan/data/output/orders_enriched.parquet"

df_parquet = spark.read.parquet(parquet_path)

# Vérification du nombre de lignes (en comparant avec df_orders_enriched si elle est en mémoire, ou affichage direct)
print(f"Nombre de lignes relues depuis le Parquet : {df_parquet.count()}")

# 3. Affichage du schéma pour observer la préservation des types
df_parquet.printSchema()

Nombre de lignes relues depuis le Parquet : 893
root
 |-- product_id: integer (nullable = true)
 |-- shipper_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- sous_total: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)
 |-- is_shipped: boolean (nullable = true)
 |-- customer_company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)


In [4]:
# Q34 — Comparer CSV vs Parquet - Comparer la taille du fichier CSV original vs le fichier Parquet. Observer le gain de compression. Expliquer pourquoi Parquet est plus efficace.
!du -sh /home/jovyan/data/raw/ /home/jovyan/data/output/orders_enriched.parquet

"""
Pour répondre à la deuxième partie de la Q34 concernant l'explication du gain de compression et l'efficacité de Parquet :

Observation du gain
Le dossier source pèse 200 Ko pour des fichiers CSV textuels éclatés et non compressés, tandis que 
le fichier Parquet consolidé (qui contient pourtant l'ensemble des données jointes et enrichies) ne pèse que 80 Ko, 
soit une réduction significative de la taille malgré l'enrichissement des données.

Pourquoi Parquet est plus efficace

Stockage en colonnes (Columnar Storage) : Contrairement au CSV qui enregistre les données ligne par ligne, 
Parquet stocke les valeurs colonne par colonne. Les données d'une même colonne étant du même type, elles se prêtent beaucoup mieux aux algorithmes de compression.

Algorithmes de compression avancés : Il utilise par défaut des méthodes performantes comme Snappy, combinées à un encodage par dictionnaire
(idéal pour les colonnes répétitives comme les pays ou les catégories) et un encodage de longueur de plage (RLE).

Préservation native des types et des métadonnées : Chaque fichier intègre des métadonnées et des statistiques par bloc, ce qui permet à Spark d'appliquer du predicate pushdown
et du column pruning (ignorer directement les blocs ou les colonnes inutiles lors d'une requête), là où le CSV nécessite une lecture textuelle séquentielle complète.
"""


200K	/home/jovyan/data/raw/
80K	/home/jovyan/data/output/orders_enriched.parquet


In [5]:
# Q35 — Partitionnement - Écrire le DataFrame partitionné par country avec partitionBy('country'). Observer la structure des dossiers créés
output_partitioned_path = "/home/jovyan/data/output/orders_partitioned.parquet"

df_parquet.write \
    .mode("overwrite") \
    .partitionBy("ship_country") \
    .parquet(output_partitioned_path)

# Affichage de la structure des dossiers créés
!ls -R /home/jovyan/data/output/orders_partitioned.parquet

/home/jovyan/data/output/orders_partitioned.parquet:
'ship_country=Argentina'  'ship_country=Germany'   'ship_country=Sweden'
'ship_country=Austria'	  'ship_country=Ireland'   'ship_country=Switzerland'
'ship_country=Belgium'	  'ship_country=Italy'	   'ship_country=UK'
'ship_country=Brazil'	  'ship_country=Mexico'    'ship_country=USA'
'ship_country=Canada'	  'ship_country=Norway'    'ship_country=Venezuela'
'ship_country=Denmark'	  'ship_country=Poland'     _SUCCESS
'ship_country=Finland'	  'ship_country=Portugal'
'ship_country=France'	  'ship_country=Spain'

'/home/jovyan/data/output/orders_partitioned.parquet/ship_country=Argentina':
part-00000-dd949c4b-4572-4273-88d2-dde9eda91c47.c000.snappy.parquet

'/home/jovyan/data/output/orders_partitioned.parquet/ship_country=Austria':
part-00000-dd949c4b-4572-4273-88d2-dde9eda91c47.c000.snappy.parquet

'/home/jovyan/data/output/orders_partitioned.parquet/ship_country=Belgium':
part-00000-dd949c4b-4572-4273-88d2-dde9eda91c47.c000.snappy.parqu

In [7]:
# Q36 — Chargement PostgreSQL via JDBC - Ajouter PostgreSQL au docker-compose. Écrire df_orders_enriched dans PostgreSQL via JDBC dans une table orders_enriched.
jdbc_url = "jdbc:postgresql://postgres:5432/tradecorp"
jdbc_props = {
    "user": "tradecorp",
    "password": "tradecorp",
    "driver": "org.postgresql.Driver"
}

df_parquet.write \
    .mode("overwrite") \
    .jdbc(url=jdbc_url, table="orders_enriched", properties=jdbc_props)
